# Lab Task 1: Titanic Survival Prediction (Decision Tree)
Runs directly in Google Colab: *Runtime > Run all*. The dataset is downloaded automatically (public mirror of Kaggle's Titanic `train.csv`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---- Load data ----
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv(url)
except Exception:
    # Fallback: upload Kaggle's train.csv manually
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up.keys())[0])

print(df.shape)
df.head()

In [ ]:
# ---- Preprocessing ----
df = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

print(df.isnull().sum().sum(), "missing values left")
df.head()

In [ ]:
# ---- Split ----
X = df.drop("Survived", axis=1)
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ---- Train Decision Tree ----
model = DecisionTreeClassifier(criterion="gini", max_depth=4, random_state=42)
model.fit(X_train, y_train)

# ---- Score ----
print("Training accuracy:", model.score(X_train, y_train))
print("Test accuracy (model score):", model.score(X_test, y_test))

y_pred = model.predict(X_test)
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

In [ ]:
# ---- Visualize the tree ----
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=X.columns, class_names=["Died", "Survived"],
          filled=True, rounded=True, fontsize=9)
plt.show()

# Feature importance
pd.Series(model.feature_importances_, index=X.columns).sort_values().plot(kind="barh", title="Feature importance")
plt.show()

In [ ]:
# ---- (Optional) find best depth ----
for d in range(1, 11):
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    print(f"max_depth={d:2d}  train={m.score(X_train,y_train):.3f}  test={m.score(X_test,y_test):.3f}")